# Segunda passada — fluxo de codificação

Uma variável por vez, com o critério do codebook antes da resposta.

**O que este notebook não faz.** Ele não sugere código, nível ou nota. Ele
localiza: verifica os documentos, conta os doze termos do §3 e mostra onde cada
um aparece. A decisão é sua em todos os campos.

**O que ele não vê.** As codificações da primeira passada não estão aqui e não
são alcançáveis daqui. Este notebook lê dois repositórios públicos — o
instrumento e o corpus congelado — e nada mais.

**Por que uma variável por vez.** Na página, as nove ficam na mesma tela, e
quem lê a V9 antes de decidir a V1 já viu para onde o serviço aponta. É a mesma
razão pela qual a âncora do piloto que nomeia o serviço em tela fica omitida:
consultar a referência é sua escolha, ter a resposta no campo de visão durante
a decisão não é.

Rode as células de cima para baixo.

## 1 · Ambiente

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    for repo in ("experimented-without-consent", "experimented-without-consent-corpus"):
        if not Path(repo).exists():
            subprocess.run(
                ["git", "clone", "--depth", "1", f"https://github.com/mrcsvg/{repo}.git"],
                check=True,
            )
    INSTRUMENT = Path("experimented-without-consent")
    CORPUS = Path("experimented-without-consent-corpus")
else:
    INSTRUMENT = Path.cwd().parent
    CORPUS = INSTRUMENT.parent / "experimented-without-consent-corpus"

sys.path.insert(0, str(INSTRUMENT / "analysis"))
import patterns as P
import codebook as C
import coding_flow as F

print(f"instrumento : {INSTRUMENT.resolve()}")
print(f"corpus      : {CORPUS.resolve()}")
print(f"codebook    : {len(C.VARIAVEIS)} variáveis · {len(C.SERVICOS)} serviços")
print(f"padrões §3  : {len(P.PATTERNS)} termos")

## 2 · Integridade e dossiê

O corpus foi congelado de vantagem da UE e cada documento carrega o hash do seu
próprio texto. Se um arquivo não bater com o hash, ele fica **fora** das
contagens em vez de contribuir com um zero silencioso — um documento ilegível e
um documento que não menciona experimentação produzem o mesmo zero, e só o
segundo é um achado.

In [ ]:
import hashlib, json

manifest = P.load_manifest(CORPUS)
QUARENTENA = []
for servico, entradas in manifest["services"].items():
    for e in entradas:
        corpo = P.strip_provenance_header((CORPUS / e["file"]).read_text(encoding="utf-8"))
        if hashlib.sha256(corpo.encode("utf-8")).hexdigest() != e["sha256"]:
            QUARENTENA.append((servico, e["file"]))

n = sum(len(v) for v in manifest["services"].values())
print(f"corpus congelado em {manifest['frozen_at']}  ·  vantagem {manifest['vantage']}")
print(f"documentos: {n}  ·  íntegros: {n - len(QUARENTENA)}  ·  quarentena: {len(QUARENTENA)}")
for servico, arq in QUARENTENA:
    print(f"  ⚠ {servico}: {Path(arq).name}")

DOSSIE = json.loads((INSTRUMENT / "analysis" / "keyword-hits.json").read_text(
    encoding="utf-8"))["servicos"]
print(f"\ndossiê §3 carregado: {len(DOSSIE)} serviços")

## 3 · Onde o progresso é gravado

O mesmo lugar que o instrumento em HTML usa: cada gravação vira um commit no
branch de dados do projeto. Você pode alternar entre os dois sem perder nada.

Grava-se primeiro em disco (`coder2-local.json`, ao lado deste notebook) e só
depois na rede. Se a sessão do Colab cair, o que já foi codificado está no
arquivo — baixe-o pelo painel de arquivos à esquerda se quiser uma cópia.

Se a rede falhar, o fluxo continua funcionando e o estado fica local; a célula
de exportação, no fim, gera o JSON para enviar por e-mail.

In [ ]:
ESTADO = F.Estado(cache=Path("coder2-local.json"))

print("servidor    :", "sincronizado" if ESTADO.online else "OFFLINE — progresso só local")
print("registros   :", len(ESTADO.records))

feitos = []
for s in C.SERVICOS:
    f_ = F.Fluxo(s, ESTADO, dossie=DOSSIE)
    feitas, total = f_.progresso()
    feitos.append((s, feitas, total))

print(f"\n{'serviço':24} {'fechadas':>9}")
for s, feitas, total in feitos:
    marca = "✓" if feitas == total else (" " if feitas == 0 else "·")
    print(f"{marca} {s:22} {feitas:>4}/{total}")
print(f"\nconcluídos: {sum(1 for _, a, b in feitos if a == b)} de {len(feitos)}")

## 4 · Escolha o serviço

Um serviço por sessão é o suficiente; o progresso fica salvo se você parar no meio.

In [ ]:
SERVICO = C.SERVICOS[0]   # troque aqui, ou use o menu abaixo

try:
    import ipywidgets as W
    from IPython.display import display
    _menu = W.Dropdown(options=C.SERVICOS, value=SERVICO, description="serviço:",
                       style={"description_width": "initial"})
    def _troca(ev):
        global SERVICO
        SERVICO = ev["new"]
        print(f"serviço agora: {SERVICO} — rode a próxima célula")
    _menu.observe(_troca, names="value")
    display(_menu)
except ImportError:
    print("ipywidgets ausente — edite SERVICO na linha acima")

print(f"serviço: {SERVICO}")

## 5 · Codificar

Para avançar, a variável precisa de **resposta e evidência**. Não é rigor
decorativo: o achado central deste estudo é uma coluna de "No", e um "No" sem
evidência é exatamente o zero não-auditável que o paper critica.

Quando a ausência for do tipo "o registro nunca contempla o tema", **isso já é a
evidência** — escreva assim e o portão abre. O que ele recusa é o campo vazio.

Voltar para rever o que você já respondeu é permitido. Pular adiante, não.

No último passo (KW) há um botão que insere a contagem já calculada, e um painel
por documento com cada ocorrência no contexto. A contagem é **crua**: `test`
conta o `testing` de QA junto com o teste A/B, `trial` conta "free trial". A
triagem é sua — o botão preenche, você corrige.

In [ ]:
import html as _html
import ipywidgets as W
from IPython.display import display

FLUXO = F.Fluxo(SERVICO, ESTADO, dossie=DOSSIE)

CSS_OK   = "padding:8px 12px;border-left:3px solid #2a7;background:#f2fbf6;border-radius:3px"
CSS_FALTA= "padding:8px 12px;border-left:3px solid #c93;background:#fff8ef;border-radius:3px"
CSS_CRIT = "padding:10px 14px;background:#f7f7f9;border-radius:4px;line-height:1.5"


class Painel:
    """Uma variável por vez. Nunca exibe código sugerido, nível ou nota."""

    def __init__(self, fluxo):
        self.f = fluxo
        self.cabecalho = W.HTML()
        self.criterio = W.HTML()
        self.corpo = W.VBox()
        self.portao = W.HTML()
        self.status = W.HTML()
        self.b_voltar = W.Button(description="◀ voltar", layout=W.Layout(width="110px"))
        self.b_avancar = W.Button(description="avançar ▶", button_style="primary",
                                  layout=W.Layout(width="130px"))
        self.b_voltar.on_click(self._voltar)
        self.b_avancar.on_click(self._avancar)
        self.widgets = {}
        self.caixa = W.VBox([self.cabecalho, self.criterio, self.corpo, self.portao,
                             W.HBox([self.b_voltar, self.b_avancar, self.status])])
        self.pintar()

    # ------------------------------------------------------------- montagem
    def _campo(self, c):
        valor = self.f.rec.get(c.chave)
        est = {"description_width": "initial"}
        larg = W.Layout(width="98%")
        if c.tipo == "select":
            w = W.Dropdown(options=c.opcoes, description=c.rotulo, style=est, layout=larg,
                           value=valor if valor in c.opcoes else c.opcoes[0])
        elif c.tipo == "checks":
            w = W.SelectMultiple(options=c.opcoes, description=c.rotulo, style=est,
                                 rows=min(len(c.opcoes), 6),
                                 value=tuple(v for v in (valor or []) if v in c.opcoes),
                                 layout=larg)
        elif c.tipo == "text":
            w = W.Textarea(value=valor or "", placeholder=c.placeholder, description=c.rotulo,
                           style=est, layout=W.Layout(width="98%", height="110px"))
        else:
            w = W.Text(value=valor or "", placeholder=c.placeholder, description=c.rotulo,
                       style=est, layout=larg)
        w.observe(lambda ev, k=c.chave: self._mudou(k, ev["new"]), names="value")
        self.widgets[c.chave] = w
        return w

    def _evidencia(self):
        """Painel por documento: contagem + cada ocorrência no contexto."""
        paineis, titulos = [], []
        for d in self.f.evidencia():
            partes = [f"<div style='font:12px/1.5 monospace;color:#555;margin-bottom:8px'>"
                      f"{_html.escape(d.get('log_line', ''))}</div>"]
            for h in d["hits"]:
                marca = P.PATTERNS.get(h["term"], {}).get("flag")
                aviso = (f"<div style='color:#a60;font-size:12px;margin-left:14px'>⚠ "
                         f"{_html.escape(marca)}</div>") if marca else ""
                partes.append(
                    f"<div style='margin:7px 0'><b>[{_html.escape(h['term'])}]</b> "
                    f"…{_html.escape(h['kwic'])}…{aviso}</div>")
            if not d["hits"]:
                partes.append("<i>nenhum dos doze termos aparece neste documento</i>")
            paineis.append(W.HTML("".join(partes)))
            titulos.append(f"{Path(d['file']).name} — {sum(d['counts'].values())} hits")
        acc = W.Accordion(children=paineis)
        for i, t in enumerate(titulos):
            acc.set_title(i, t)
        acc.selected_index = None
        return acc

    def _inserir_log(self, _):
        w = self.widgets.get("keyword_log")
        if w is not None:
            w.value = self.f.log_sugerido()

    def pintar(self):
        p = self.f.atual()
        feitas, total = self.f.progresso()
        self.cabecalho.value = (
            f"<div style='margin-bottom:6px'><span style='font:600 13px sans-serif;"
            f"background:#333;color:#fff;padding:2px 7px;border-radius:3px'>{p.vid}</span> "
            f"<b style='font-size:17px'>{_html.escape(p.titulo)}</b></div>"
            f"<div style='color:#666;font-size:13px'>{_html.escape(self.f.servico)} · "
            f"passo {self.f.i + 1} de {len(self.f.passos)} · {feitas} fechadas</div>"
            f"<div style='margin-top:8px;font-size:14px'>{p.regra}</div>")

        ancoras = ""
        if p.ancoras:
            itens = "".join(f"<li>{a}</li>" for a in p.ancoras)
            ancoras = f"<div style='margin-top:10px'><b>Âncoras do piloto</b><ul>{itens}</ul></div>"
        if p.ancoras_suprimidas:
            n = len(p.ancoras_suprimidas)
            ancoras += (f"<div style='margin-top:6px;color:#666;font-size:13px'>"
                        f"{'Âncoras foram' if n > 1 else 'Uma âncora foi'} <b>omitida</b> aqui "
                        f"porque nomeia o serviço que você está codificando. Ela está no "
                        f"codebook, se quiser consultá-la deliberadamente.</div>")
        self.criterio.value = f"<div style='{CSS_CRIT}'>{p.criterio}{ancoras}</div>"

        self.widgets = {}
        filhos = [self._campo(c) for c in p.campos]
        if p.vid == "KW":
            b = W.Button(description="inserir a contagem calculada", icon="clipboard",
                         layout=W.Layout(width="260px"))
            b.on_click(self._inserir_log)
            filhos += [
                W.HTML("<div style='color:#666;font-size:13px;margin-top:4px'>A contagem é "
                       "crua. <code>test</code> inclui o <i>testing</i> de QA; <code>trial</code> "
                       "inclui <i>free trial</i>. Abra os documentos abaixo e corrija.</div>"),
                b, self._evidencia()]
        self.corpo.children = tuple(filhos)
        self.b_voltar.disabled = self.f.i == 0
        self._portao()

    # ----------------------------------------------------------- interação
    def _mudou(self, chave, novo):
        self.f.responder({chave: list(novo) if isinstance(novo, tuple) else novo}, rede=False)
        self._portao()

    def _portao(self):
        pend = self.f.faltando()
        self.b_avancar.disabled = bool(pend)
        if pend:
            itens = " · ".join(f"<code>{k}</code> <span style='color:#888'>({p})</span>"
                               for k, p in pend)
            self.portao.value = f"<div style='{CSS_FALTA}'>Falta: {itens}</div>"
        else:
            self.portao.value = (f"<div style='{CSS_OK}'>Variável fechada — pode avançar.</div>")

    def _voltar(self, _):
        self.f.voltar()
        self.pintar()

    def _avancar(self, _):
        online = self.f.estado.gravar(self.f.servico, self.f.rec)
        self.status.value = ("<span style='color:#2a7'>salvo no servidor</span>" if online
                             else "<span style='color:#c93'>salvo localmente</span>")
        if not self.f.avancar():
            feitas, total = self.f.progresso()
            if feitas == total:
                self.f.rec["done"] = True
                self.f.estado.gravar(self.f.servico, self.f.rec)
                self.portao.value = (f"<div style='{CSS_OK}'><b>{_html.escape(self.f.servico)} "
                                     f"concluído.</b> Volte à célula 4, escolha o próximo "
                                     f"serviço e rode esta célula de novo.</div>")
            self.b_avancar.disabled = True
            return
        self.pintar()

    def mostrar(self):
        display(self.caixa)


Painel(FLUXO).mostrar()

## 6 · Exportar

Gera o JSON no mesmo formato que o `compute-agreement.py` consome. Se o servidor
estiver sincronizado isto é redundante — mas custa nada e serve de backup.

In [ ]:
import json
from datetime import datetime, timezone

registros = {s: ESTADO.registro(s) for s in C.SERVICOS if ESTADO.registro(s)}
saida = [dict(r, service=s) for s, r in registros.items()]

destino = Path("coder2-export.json")
destino.write_text(json.dumps(saida, ensure_ascii=False, indent=1), encoding="utf-8")

completos = sum(1 for s in C.SERVICOS if F.Fluxo(s, ESTADO, dossie=DOSSIE).progresso()[0] == len(C.VARIAVEIS))
print(f"{destino}  ·  {len(saida)} serviços iniciados  ·  {completos} completos")
print(f"servidor: {'sincronizado' if ESTADO.online else 'OFFLINE — envie este arquivo'}")
print("\nBaixe pelo painel de arquivos à esquerda (ícone de pasta).")